# Practical 2: Classical machine learning with scikit-learn

**Intro to AI, Machine Learning and Data Science** · about 20 minutes

Can we tell which grape cultivar a wine came from using only its chemistry? The *wine* dataset contains 13 chemical measurements (alcohol, flavanoids, colour intensity, proline, ...) for 178 wines from three cultivars grown in the same region of Italy.

You will:

1. **Explore** the data and set up an honest train/test split
2. Build a **baseline**, then a real model inside a **pipeline**
3. See why **feature scaling** matters for some models and not others
4. **Compare models** with cross-validation and **tune** hyperparameters
5. Explore the data **without labels**: PCA and k-means clustering
6. Watch **data leakage** manufacture a result out of pure noise

Sections 1 to 4 and 6 are the core path; section 5 is short, and the *Bonus* cells are for fast finishers. Solutions are hidden under *Show solution*.

*Data:* Forina, M. et al. (1988), UCI Machine Learning Repository, [doi:10.24432/C5PC7J](https://doi.org/10.24432/C5PC7J). Bundled with scikit-learn, so no download is needed.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]),
})
print("scikit-learn", sklearn.__version__)

## 1. Explore and split

In scikit-learn, data is a **feature matrix** `X` (one row per sample, one column per feature) and a **target vector** `y` (one label per sample).

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
print("X shape:", X.shape)
print("samples per cultivar:", y.value_counts().sort_index().to_dict())
X.head()

In [ ]:
X.describe().T[["mean", "std", "min", "max"]].round(2)

Notice the scales: `proline` is in the hundreds to thousands, `hue` is around 1. Keep that in mind.

Before doing anything else, we lock away a **test set** (25 %) that we will only touch once, at the very end. `stratify=y` keeps the class proportions the same in both parts, and `random_state` makes the split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
print(f"train: {X_train.shape}, test: {X_test.shape}")

## 2. Baseline first, then a model

A **baseline** tells you what "no skill" looks like. Here: always predict the most common cultivar. Any real model must beat this.

Every scikit-learn model follows the same API: create it, `.fit(X, y)` on training data, then `.predict(X)` or `.score(X, y)`.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"baseline test accuracy: {baseline.score(X_test, y_test):.3f}")

Now a real model: **k-nearest neighbours** (kNN) predicts the majority class among the *k* most similar training wines, measured by Euclidean distance. We try it on the raw features and inside a **pipeline** that first standardises each feature to mean 0 and standard deviation 1.

A `Pipeline` chains preprocessing and model into one object. When you call `.fit`, the scaler learns its means and standard deviations **from the training data only**, which is exactly what prevents leakage later.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

knn_raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)

print(f"kNN on raw features   : {knn_raw.score(X_test, y_test):.3f}")
print(f"kNN on scaled features: {knn_scaled.score(X_test, y_test):.3f}")

### Your turn

Why does scaling make such a big difference for kNN? *Hint:* compute the standard deviation of each column of `X_train` and think about what dominates a Euclidean distance.

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
X_train.std().sort_values(ascending=False).round(2).head()
```

`proline` has a standard deviation of about 300, while most other features are below 5. In a Euclidean distance, differences in proline swamp everything else, so unscaled kNN is effectively a one-feature model. Distance-based and gradient-based methods (kNN, SVMs, linear models trained by gradient descent, neural networks) need scaled inputs. Tree-based models split one feature at a time and do not care.

</details>

Accuracy is a single number. A **confusion matrix** shows *which* classes get confused with which.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ConfusionMatrixDisplay.from_estimator(knn_scaled, X_test, y_test, cmap="Blues", colorbar=False)
plt.grid(False)
print(classification_report(y_test, knn_scaled.predict(X_test), target_names=wine.target_names))

## 3. Compare models with cross-validation

With only 133 training wines, one lucky or unlucky split can mislead us. **k-fold cross-validation** splits the *training* set into 5 folds, trains on 4 and validates on the 5th, rotating 5 times. We get a mean and a spread, and the test set stays untouched.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC

models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "k-nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "SVM (RBF kernel)": make_pipeline(StandardScaler(), SVC()),
    "random forest": RandomForestClassifier(random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scores = {name: cross_val_score(model, X_train, y_train, cv=cv) for name, model in models.items()}
summary = pd.DataFrame(scores).agg(["mean", "std"]).T.sort_values("mean", ascending=False)
summary.round(3)

On this small, clean dataset several models are essentially tied: differences smaller than the standard deviation are noise. In practice, prefer the **simplest** model among the tied ones: it is faster, easier to explain and less likely to overfit.

### Your turn

Add a `DecisionTreeClassifier(random_state=0)` to the comparison. How does a single tree compare with the random forest (an average of many trees)?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
from sklearn.tree import DecisionTreeClassifier

tree_scores = cross_val_score(DecisionTreeClassifier(random_state=0), X_train, y_train, cv=cv)
print(f"decision tree: {tree_scores.mean():.3f} +/- {tree_scores.std():.3f}")
```

The single tree scores lower (about 0.94 against 0.97 for the forest). A deep tree memorises its training data; averaging many randomised trees (a random forest) cancels much of that variance. This is the idea behind **ensembles**, and the gap is usually much larger on noisier data than on this clean dataset.

</details>

## 4. Tune hyperparameters, then test once

**Hyperparameters** are settings you choose rather than learn, e.g. the SVM's regularisation strength `C` and kernel width `gamma`. `GridSearchCV` tries every combination with cross-validation on the training set and refits the best one.

Pipeline steps are named after their class in lowercase, so the SVM's `C` is addressed as `svc__C`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.01, 0.1, 1],
}
search = GridSearchCV(make_pipeline(StandardScaler(), SVC()), param_grid, cv=cv, n_jobs=4)
search.fit(X_train, y_train)

print("best hyperparameters:", search.best_params_)
print(f"best cross-validation accuracy: {search.best_score_:.3f}")

Only now, with every choice made, do we open the drawer and evaluate **once** on the test set. This number is our honest estimate of performance on new wines.

In [ ]:
print(f"test accuracy of the tuned SVM: {search.score(X_test, y_test):.3f}")

> `n_jobs=4` runs 4 fits in parallel on 4 CPU cores. A grid search is *embarrassingly parallel*: every combination is independent. On MeluXina you can spread thousands of such fits over many cores or many jobs; you will do exactly that in Practical 3.

### Bonus: which features matter?

**Permutation importance** shuffles one feature at a time on the test set and measures how much the accuracy drops.

In [ ]:
from sklearn.inspection import permutation_importance

forest = RandomForestClassifier(random_state=0).fit(X_train, y_train)
result = permutation_importance(forest, X_test, y_test, n_repeats=20, random_state=0)
importance = pd.Series(result.importances_mean, index=X.columns).sort_values()
ax = importance.tail(8).plot.barh(color="#2a78d6", width=0.6)
ax.set(xlabel="drop in accuracy when shuffled", title="Permutation importance (random forest)");

Treat importances as a *hypothesis generator*, not as proof of mechanism. When features are correlated (as many chemical measurements are), the model can use either one, and importance gets shared or hidden.

## 5. Unsupervised learning: structure without labels

Suppose we had no cultivar labels. Could we still discover groups? We standardise the features, then:

* **PCA** (principal component analysis) rotates the 13-dimensional data onto the directions of largest variance. These are the eigenvectors of the covariance matrix, so we can plot the data in 2D.
* **k-means** partitions the samples into *k* clusters by minimising the distance of each sample to its cluster centre.

Neither method ever sees `y`. We only use it afterwards to check what they found.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(X_scaled)
Z = pca.transform(X_scaled)
print("variance explained by PC1, PC2:", pca.explained_variance_ratio_.round(3))

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_scaled)
print(f"agreement between clusters and cultivars (adjusted Rand index): "
      f"{adjusted_rand_score(y, kmeans.labels_):.3f}")

# k-means numbers its clusters arbitrarily. For the plot only, name each cluster
# after the cultivar most common in it, so that matching groups get matching colours.
y_np = y.to_numpy()
majority = {c: np.bincount(y_np[kmeans.labels_ == c]).argmax() for c in range(3)}
cluster_as_cultivar = np.array([majority[c] for c in kmeans.labels_])

colours = np.array(["#2a78d6", "#eb6834", "#1baf7a"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, labels, title in [(axes[0], y_np, "coloured by true cultivar"),
                          (axes[1], cluster_as_cultivar, "coloured by k-means cluster")]:
    ax.scatter(Z[:, 0], Z[:, 1], c=colours[labels], s=30, edgecolor="white", linewidth=0.8)
    ax.set(title=title, xlabel="PC1", ylabel="PC2")

An adjusted Rand index of 1 means perfect agreement and 0 means chance. The clusters found **without labels** match the cultivars closely: the chemistry really does separate them. The few points whose colour differs between the panels are the wines k-means put in the "wrong" group.

In real research you rarely know the "right" number of clusters.

### Your turn

Run k-means for $k = 2, \dots, 6$ and compute the **silhouette score** (`sklearn.metrics.silhouette_score`; higher is better) for each. Which *k* does it prefer?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
from sklearn.metrics import silhouette_score

for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_scaled)
    print(f"k = {k}: silhouette = {silhouette_score(X_scaled, labels):.3f}")
```

The silhouette score peaks at $k = 3$, matching the three cultivars. On messier data the answer is rarely this clean; treat such scores as guidance and check that clusters make scientific sense.

</details>

## 6. Data leakage: getting over 80 % accuracy from pure noise

This is the most important cell of the practical. We create a dataset of **pure random noise**: 100 "samples" with 5,000 random "features" (think thousands of monitoring metrics) and random binary labels. No model can do better than 50 % on new data.

A common workflow is: select the features most correlated with the label, then cross-validate a classifier on them. Let's do that in the **wrong** order and then the **right** one.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)

# WRONG: feature selection sees ALL the data, including the future validation folds
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_selected, y_noise, cv=5)

# RIGHT: feature selection is part of the pipeline, re-fitted inside every training fold
pipeline = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(pipeline, X_noise, y_noise, cv=5)

print(f"wrong order: {wrong.mean():.2f} accuracy on pure noise")
print(f"right order: {right.mean():.2f} accuracy on pure noise")

The wrong version "finds" a strong signal in random numbers, because the 20 features were chosen for looking predictive **on the very samples used to validate them**. This mistake has been found in hundreds of published papers across many fields (Kapoor & Narayanan, *Patterns* 2023).

**Rules of thumb**

* Split first. Every step that learns from data (scaling, imputation, feature selection, PCA) goes **inside** the pipeline.
* Split in the way the model will be used: by time, by user, by machine, by site (`GroupKFold`, `TimeSeriesSplit`).
* If a result looks too good to be true, look for leakage first.

## Bonus: a different dataset

`sklearn.datasets.load_breast_cancer` has 30 features computed from cell-nucleus images for 569 tumours, labelled benign or malignant. Reuse your model comparison from section 3 on it. Which metric would you care about most here, and why is accuracy not enough?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import cross_validate

Xb, yb = load_breast_cancer(return_X_y=True)   # 0 = malignant, 1 = benign
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scoring = {"accuracy": "accuracy",
           "malignant_recall": make_scorer(recall_score, pos_label=0)}
res = cross_validate(model, Xb, yb, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})
```

Missing a malignant tumour (false negative) is far worse than a false alarm, so you would care most about **recall of the malignant class** (sensitivity), and choose the decision threshold accordingly. Pick metrics that reflect the cost of each kind of error in *your* application.

</details>

## Summary

* Start with a **baseline**; judge models on **held-out** data.
* The scikit-learn API is always `fit` / `predict` / `score`; **pipelines** bundle preprocessing with the model.
* **Scale** features for distance- and gradient-based models.
* Use **cross-validation** on the training set to compare and tune; touch the test set **once**.
* Unsupervised methods (**PCA**, **k-means**) reveal structure without labels.
* **Data leakage** can create convincing results out of nothing. Split first and keep every learned step inside the pipeline.

Next: Part 3, neural networks.